# 22 — Fock-Resolved Experiments

Consolidates legacy experiments 65, 66, and 67.

1. **Fock-resolved Ramsey** — cavity-state-selective Ramsey fringe measurement (Exp 65)
2. **Fock-resolved Power Rabi** — cavity-state-selective Rabi oscillations (Exp 66)
3. **T2 Ramsey on cavity** — qubit T2 Ramsey conditioned on cavity state (Exp 67)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    FockResolvedPowerRabi,
    FockResolvedRamsey,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="22_fock_resolved_experiments",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

storage_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="21_storage_cavity_characterization",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if storage_checkpoint is not None:
    print(f"Prior stage 21 status: {storage_checkpoint['status']}")

## 2. Fock-Resolved Defaults

In [ ]:
FOCK_RAMSEY_N_AVG = 10000
FOCK_RAMSEY_MAX_N = 5

FOCK_RABI_N_AVG = 10000
FOCK_RABI_MAX_N = 5

CAVITY_T2_N_AVG = 10000

print("Fock-resolved experiment settings:")
print(f"  Ramsey: max_n={FOCK_RAMSEY_MAX_N}, n_avg={FOCK_RAMSEY_N_AVG}")
print(f"  Power Rabi: max_n={FOCK_RABI_MAX_N}, n_avg={FOCK_RABI_N_AVG}")
print(f"  Cavity T2 Ramsey n_avg: {CAVITY_T2_N_AVG}")

## 3. Fock-Resolved Ramsey — Exp 65

Ramsey fringe measurement conditioned on cavity Fock state |n⟩.

In [ ]:
fock_ramsey = FockResolvedRamsey(session)
fock_ramsey_result = fock_ramsey.run(
    max_n=FOCK_RAMSEY_MAX_N,
    n_avg=FOCK_RAMSEY_N_AVG,
)
fock_ramsey_analysis = fock_ramsey.analyze(fock_ramsey_result, update_calibration=False)
fock_ramsey.plot(fock_ramsey_analysis)

print("Fock-resolved Ramsey complete.")
if hasattr(fock_ramsey_analysis, "metrics"):
    for k, v in fock_ramsey_analysis.metrics.items():
        print(f"  {k}: {v}")

## 4. Fock-Resolved Power Rabi — Exp 66

Power Rabi oscillations conditioned on cavity Fock state |n⟩.

In [ ]:
fock_rabi = FockResolvedPowerRabi(session)
fock_rabi_result = fock_rabi.run(
    max_n=FOCK_RABI_MAX_N,
    n_avg=FOCK_RABI_N_AVG,
)
fock_rabi_analysis = fock_rabi.analyze(fock_rabi_result, update_calibration=False)
fock_rabi.plot(fock_rabi_analysis)

print("Fock-resolved Power Rabi complete.")
if hasattr(fock_rabi_analysis, "metrics"):
    for k, v in fock_rabi_analysis.metrics.items():
        print(f"  {k}: {v}")

## 5. T2 Ramsey on Cavity — Exp 67

Qubit T2 Ramsey conditioned on the cavity photon number.

In [ ]:
# This uses a displaced Fock-resolved Ramsey to extract cavity-dependent T2
cavity_t2_result = session.cavity_conditional_t2_ramsey(
    n_avg=CAVITY_T2_N_AVG,
)

print("Cavity-conditioned T2 Ramsey complete.")

## 6. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="22_fock_resolved_experiments",
    status="characterized",
    summary="Fock-resolved Ramsey, Power Rabi, and cavity-conditioned T2 characterized.",
    consumed_inputs={
        "fock_ramsey_max_n": FOCK_RAMSEY_MAX_N,
        "fock_ramsey_n_avg": FOCK_RAMSEY_N_AVG,
        "fock_rabi_max_n": FOCK_RABI_MAX_N,
        "fock_rabi_n_avg": FOCK_RABI_N_AVG,
        "cavity_t2_n_avg": CAVITY_T2_N_AVG,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="23_quantum_state_preparation",
    notes=[
        "All Fock-resolved experiments are characterization-only.",
        "Cavity-conditioned T2 (Exp 67) uses session.cavity_conditional_t2_ramsey.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")